# k-means clustering on 38 economies

1. How does k-means form groups from economies without labels?
2. Why must features share one scale, and what does one far economy do?
3. How good is a grouping, and how many groups should there be?
4. What does a group's centroid say about its economies?

k-means runs first by hand on six toy economies, then on 38 economies of the BIS Data Portal, grouped by three features of 2025: the mean policy rate, the mean CPI inflation and the change of the policy rate from December 2021 to December 2025. Türkiye is reported as a group of its own, and the scaler is fitted on the other 37.

Policy rates and consumer prices: BIS Data Portal. Snapshot 30 September 2026.

In [ ]:
import pathlib

import pandas as pd

DATA_URL = "https://digital-ai-finance.github.io/data-science-msc/data/"
SNAPSHOT = "2026-09-30"


def load(name):
    """Read a snapshot by name, e.g. load("bis-economies")."""
    local = pathlib.Path("../data/market") / f"{SNAPSHOT}-{name}.csv"
    if local.exists():  # run from material/notebooks/ before the files are published
        return pd.read_csv(local)
    url = f"{DATA_URL}{SNAPSHOT}-{name}.csv"
    try:
        return pd.read_csv(url)
    except Exception as err:
        raise RuntimeError(f"{url} not reachable ({err}). Download the file in a browser, upload "
                           "it in the Colab file panel and read it with pd.read_csv.") from err

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.preprocessing import StandardScaler

SEED = 42  # every KMeans passes n_init=10 and random_state=SEED, so a rerun gives the same groups
BLUE, ORANGE, RED, GREY = "#0066CC", "#FF7F0E", "#D62728", "#7F7F7F"

## 1. Assign and update by hand

Six toy economies with two features, policy rate and inflation in per cent, both on one scale. The run starts with c1 = E3 and c2 = E2. Each round assigns every economy to its nearest centroid, then moves each centroid to the mean of its economies. The values are stylised.

In [ ]:
toy = np.array([[2, 3], [3, 2], [2, 1], [8, 7], [9, 6], [7, 8]], float)  # E1 to E6
names = ["E1", "E2", "E3", "E4", "E5", "E6"]
centres = toy[[2, 1]].copy()  # start: c1 = E3, c2 = E2
previous = None
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharex=True, sharey=True)
for r, ax in enumerate(axes, start=1):
    labels = np.linalg.norm(toy[:, None] - centres[None], axis=2).argmin(axis=1)  # assign
    if previous is None:
        changed = "first assignment"
    else:
        changed = ", ".join(n for n, a, b in zip(names, previous, labels) if a != b) or "none"
    ax.scatter(toy[:, 0], toy[:, 1], c=[BLUE if g == 0 else RED for g in labels], s=50, zorder=3)
    ax.scatter(centres[:, 0], centres[:, 1], c=[BLUE, RED], marker="x", s=140, linewidths=3)
    for n, (x, y) in zip(names, toy):
        ax.annotate(n, (x, y), xytext=(5, 4), textcoords="offset points")
    ax.set_title(f"round {r}")
    ax.set_xlabel("policy rate, %")
    print(f"round {r}: groups {[int(g) + 1 for g in labels]}, changed: {changed}, "
          f"WCSS {((toy - centres[labels]) ** 2).sum():.2f}")
    if previous is not None and (labels == previous).all():
        print("no economy changed its group: the loop stops")
        break
    centres = np.array([toy[labels == k].mean(axis=0) for k in range(2)])  # update
    print(f"  update: c1 = ({centres[0, 0]:.2f}, {centres[0, 1]:.2f}), "
          f"c2 = ({centres[1, 0]:.2f}, {centres[1, 1]:.2f}), WCSS {((toy - centres[labels]) ** 2).sum():.2f}")
    previous = labels
axes[0].set_ylabel("inflation, %")
plt.show()

## 2. Features on one scale, and one far economy

A distance adds squared differences over all features, so the feature with the widest spread decides it. Standardising gives every feature mean 0 and sd 1.

In [ ]:
FEATURES = ["policy_rate_2025", "cpi_inflation_2025", "policy_change_2021_2025"]
econ = load("bis-economies")
print(len(econ), "economies")
print(econ.nlargest(5, "policy_rate_2025")[["economy"] + FEATURES].round(2).to_string(index=False))
summary = econ[FEATURES].describe()
summary.loc["std"] = econ[FEATURES].std(ddof=0)  # population sd, divided by n: StandardScaler and the slides use it
print(summary.rename(index={"std": "sd (ddof=0)"}).round(2))

In [ ]:
z_all = StandardScaler().fit_transform(econ[FEATURES])
tr = econ.index[econ.code == "TR"][0]
print("Türkiye in standard units among all 38:", ", ".join(f"{v:.2f}" for v in z_all[tr]))
dist = np.linalg.norm(z_all - z_all[tr], axis=1)
dist[tr] = np.inf
print(f"nearest economy: {econ.economy[dist.argmin()]}, {dist.min():.2f} standard units away")
others = np.delete(z_all, tr, axis=0)
pair = np.linalg.norm(others[:, None] - others[None], axis=2)[np.triu_indices(len(others), 1)]
print(f"median distance between two of the other 37: {np.median(pair):.2f}")
km_all = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit(z_all)
sizes = np.bincount(km_all.labels_)
print("group sizes with k = 2 on all 38:", sorted(int(n) for n in sizes))
print("alone:", ", ".join(econ.economy[km_all.labels_ == sizes.argmin()]))
print(f"WCSS with one group {((z_all - z_all.mean(axis=0)) ** 2).sum():.2f}, with two {km_all.inertia_:.2f}")

In [ ]:
rest = econ[econ.code != "TR"].reset_index(drop=True)
scaler = StandardScaler().fit(rest[FEATURES])
print("sd with Türkiye:   ", ", ".join(f"{v:.2f}" for v in StandardScaler().fit(econ[FEATURES]).scale_))
print("sd without Türkiye:", ", ".join(f"{v:.2f}" for v in scaler.scale_))
print("means of the 37:   ", ", ".join(f"{v:.2f}" for v in scaler.mean_))
print(f"policy-rate variance over inflation variance: {scaler.var_[0] / scaler.var_[1]:.1f}")
X_raw, X = rest[FEATURES].to_numpy(), scaler.transform(rest[FEATURES])
idx = {c: i for i, c in enumerate(rest.code)}
print(f"Russia: policy rate {X_raw[idx['RU'], 0]:.2f}, z among the 37 {X[idx['RU'], 0]:.2f}")

In [ ]:
for a, b in (("XM", "IN"), ("XM", "TH"), ("IN", "TH")):
    i, j = idx[a], idx[b]
    diff = X_raw[i] - X_raw[j]
    print(f"{rest.economy[i]} and {rest.economy[j]}: raw {np.linalg.norm(diff):.2f}, "
          f"policy rate {100 * diff[0] ** 2 / (diff ** 2).sum():.0f}% of the square; "
          f"standardised {np.linalg.norm(X[i] - X[j]):.2f}")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, M, title in ((axes[0], X_raw, "raw per cent"), (axes[1], X, "standard units")):
    km = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit(M)
    small = km.labels_ == np.bincount(km.labels_).argmin()
    print(f"{title}, smaller of two groups: {', '.join(sorted(rest.economy[small]))}")
    ax.scatter(M[:, 0], M[:, 1], c=np.where(small, RED, BLUE), s=25)
    for c in ("XM", "IN", "TH"):
        ax.annotate(rest.economy[idx[c]], M[idx[c], :2], xytext=(6, -12), textcoords="offset points")
    ax.set_title(f"{title}; red: the smaller group")
    ax.set_xlabel("policy rate")
    ax.set_ylabel("inflation")
plt.show()

## 3. WCSS, elbow, silhouette, starts

WCSS adds each economy's squared distance to its own centroid; scikit-learn calls it `inertia_`. The silhouette of one economy is (b − a)/max(a, b), with a its mean distance to its own group and b to the nearest other group.

In [ ]:
ks = range(1, 11)
wcss = [KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X).inertia_ for k in ks]
print("WCSS of the 37, k = 1 to 10:", ", ".join(f"{w:.2f}" for w in wcss))
print(f"two groups cut WCSS by {100 * (1 - wcss[1] / wcss[0]):.0f} per cent")
plt.plot(ks, wcss, "o-", color="black")
plt.plot(2, wcss[1], "o", color=ORANGE, ms=10)
plt.xlabel("number of groups k")
plt.ylabel("WCSS")
plt.title("Elbow of the 37 economies")
plt.show()

In [ ]:
K_RANGE = range(2, 7)
sil38 = {k: silhouette_score(z_all, KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(z_all))
         for k in K_RANGE}
sil37 = {k: silhouette_score(X, KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(X))
         for k in K_RANGE}
print("mean silhouette, all 38:", ", ".join(f"k = {k}: {s:.3f}" for k, s in sil38.items()))
print("mean silhouette, the 37:", ", ".join(f"k = {k}: {s:.2f}" for k, s in sil37.items()))
k_chosen = max(sil37, key=sil37.get)
print("k_chosen =", k_chosen)
km12 = KMeans(n_clusters=k_chosen, n_init=10, random_state=SEED).fit(X)
order = np.argsort(-km12.cluster_centers_[:, 0])  # group 1: the higher policy-rate centroid, red as in the slides
labels = np.argsort(order)[km12.labels_]
s_each = silhouette_samples(X, labels)
print("lowest silhouettes:", ", ".join(f"{rest.economy[i]} {s_each[i]:.2f}" for i in np.argsort(s_each)[:2]))
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, sil, title in ((axes[0], sil38, "all 38"), (axes[1], sil37, "the 37")):
    ax.bar(list(sil), list(sil.values()), color=GREY)
    ax.set_title(title)
    ax.set_xlabel("number of groups k")
    ax.set_ylabel("mean silhouette")
y = 0
for g in range(k_chosen):
    vals = np.sort(s_each[labels == g])
    axes[2].barh(range(y, y + len(vals)), vals, height=1.0, color=[RED, BLUE, ORANGE][g],
                 label=f"group {g + 1}: {len(vals)} economies")
    y += len(vals) + 2
axes[2].axvline(s_each.mean(), color="black", ls="--", label=f"mean {s_each.mean():.2f}")
axes[2].set_xlabel("silhouette of each economy")
axes[2].set_yticks([])
axes[2].set_title(f"k = {k_chosen}, the 37")
axes[2].legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
plt.show()

In [ ]:
starts = [KMeans(n_clusters=3, init="random", n_init=1, random_state=s).fit(X).inertia_ for s in range(10)]
print("ten single random starts, k = 3:", ", ".join(f"{w:.2f}" for w in starts))
values = sorted({round(w, 2) for w in starts})
print(f"{len(values)} different WCSS values, from {values[0]:.2f} to {values[-1]:.2f}")

## 4. Groups as profiles

`scaler.inverse_transform` turns each centroid back into per cent: mean plus z times sd, feature by feature. Group 1 is the group with the higher policy-rate centroid.

In [ ]:
km = KMeans(n_clusters=k_chosen, n_init=10, random_state=SEED).fit(X)
order = np.argsort(-km.cluster_centers_[:, 0])  # group 1: the higher policy-rate centroid
group = np.argsort(order)[km.labels_]
centres_z = km.cluster_centers_[order]
centres = scaler.inverse_transform(centres_z)
tr_row = econ.loc[econ.code == "TR", FEATURES].to_numpy()[0]
columns = ["policy rate", "inflation", "rate change"]
profile = pd.DataFrame(np.vstack([tr_row, centres]), columns=columns, index=["Türkiye", "group 1", "group 2"])
profile.insert(0, "economies", [1] + [int(n) for n in np.bincount(group)])
print(profile.round(2))
print(pd.DataFrame(centres_z, columns=columns, index=["z, group 1", "z, group 2"]).round(2))
for g in range(k_chosen):
    print(f"group {g + 1}: {', '.join(sorted(rest.economy[group == g]))}")
print("Switzerland is in group", group[idx["CH"]] + 1)
d_mx = np.linalg.norm(centres_z - X[idx["MX"]], axis=1)
print(f"Mexico: {d_mx[0]:.2f} from group 1, {d_mx[1]:.2f} from group 2")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(X_raw[:, 0], X_raw[:, 1], c=np.where(group == 0, RED, BLUE), s=30)
ax.scatter(centres[:, 0], centres[:, 1], c=[RED, BLUE], marker="x", s=140, linewidths=3)
ax.annotate(f"Türkiye ({tr_row[0]:.2f}, {tr_row[1]:.2f}) lies off the plot", xy=(20.3, 8.8),
            xytext=(11.5, 1.0), color=ORANGE, arrowprops=dict(arrowstyle="->", color=ORANGE))
ax.set_xlim(-1, 20.5)
ax.set_xlabel("policy rate 2025, %")
ax.set_ylabel("inflation 2025, %")
ax.set_title("red: group 1, blue: group 2; crosses: centroids")
plt.show()

In [ ]:
new = pd.DataFrame([[7.0, 4.0, 3.0]], columns=FEATURES)  # a stylised economy
z_new = scaler.transform(new)[0]
d_new = np.linalg.norm(centres_z - z_new, axis=1)
print("new economy in standard units:", ", ".join(f"{v:.2f}" for v in z_new))
print(f"distance to group 1 {d_new[0]:.2f}, to group 2 {d_new[1]:.2f}: it joins group {d_new.argmin() + 1}")
z_tr = scaler.transform(econ.loc[econ.code == "TR", FEATURES])[0]
d_tr = np.linalg.norm(centres_z - z_tr, axis=1)
member = np.linalg.norm(X[group == 0] - centres_z[0], axis=1).mean()
print("Türkiye in the 37's standard units:", ", ".join(f"{v:.2f}" for v in z_tr))
print(f"Türkiye lies {d_tr[0]:.2f} from group 1 and {d_tr[1]:.2f} from group 2; members of group 1 "
      f"lie {member:.2f} from its centroid on average, and Türkiye {d_tr[0] / member:.0f} times as far")

## Exercises

**E1.** Run k-means with k = 3 on the 37 and name the three groups from their centroids.

**E2.** Cluster the 37 on raw per cent and compare the groups with the seven and the thirty of the standardised run.

**E3.** Drop the rate change, recluster with k = 2, and list the economies that change group.

**E4.** Fit the scaler on all 38 and compute the mean silhouette for k = 2 to 6 again. What happens to the groups of the 37?

## Newer data (optional)

The policy rates of the latest months, fetched live from the BIS Data Portal, differ from the means of 2025 in the snapshot of 30 September 2026 that the slides use.

In [ ]:
BIS = "https://stats.bis.org/api/v1/data/{}?startPeriod={}&format=csv"


def bis(flow, start):
    """Live BIS series, e.g. bis("WS_CBPOL/M..", "2021-12")."""
    return pd.read_csv(BIS.format(flow, start))


pol = bis("WS_CBPOL/M..", "2025-10")
latest = pol.groupby("REF_AREA")["OBS_VALUE"].agg(["mean", "count"])
latest.columns = ["mean of the latest months", "months"]
compare = econ.set_index("code")[["economy", "policy_rate_2025"]].join(latest, how="left")
compare["change"] = compare["mean of the latest months"] - compare["policy_rate_2025"]
print("months from", pol["TIME_PERIOD"].min(), "to", pol["TIME_PERIOD"].max())
print(compare.round(2).sort_values("change").to_string())

## Summary

1. Each economy joins its nearest centroid, then each centroid moves to the mean of its economies; the two steps repeat until no economy changes group.
2. A distance adds squared differences, so the feature with the widest spread decides; standardising gives every feature sd 1. Türkiye lies more than five sds out and is reported alone.
3. WCSS measures how tightly economies sit around their centroids, and several starts keep the lowest. The elbow and the mean silhouette choose k: for the 37 both point at two groups.
4. A centroid is the group's average economy; mean plus z times sd turns it back into per cent. A new economy joins the nearest centroid after the same scaling.